In [1]:
import pandas as pd
import numpy as np
import ast
import re
import json
import unicodedata
from pathlib import Path

RAW_PATH = Path("data/raw/jobSkillData/all_job_post.csv")
OUT_DIR = Path("data/processed/jobSkillData")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:

# Load data
try:
    df = pd.read_csv(RAW_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(RAW_PATH, encoding="utf-8-sig")

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate job_id:", df.duplicated(subset=["job_id"]).sum())

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/jobSkillData/all_job_post.csv'

In [ ]:
# Text cleaning helpers
BULLET_RE = re.compile(r"[\u2022\u2023\u25E6\u2043\u2219\uf0b7]")

def clean_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("\r\n", "\n").replace("\r", "\n")
    x = BULLET_RE.sub(" ", x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n\s*\n+", "\n", x)
    x = re.sub(r"\s+([.,;:!?])", r"\1", x)
    return x.strip()

def normalize_skill(s):
    if pd.isna(s):
        return np.nan
    s = clean_text(s)
    if not s:
        return np.nan
    s = s.lower()
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s\-\+/#\.]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else np.nan

def parse_skill_set(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return []

    # Primary parser: stringified Python list
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            out = []
            for item in parsed:
                if item is None:
                    continue
                item = str(item).strip()
                if item and item.lower() not in {"nan", "none", "null"}:
                    out.append(item)
            return out
        if isinstance(parsed, str):
            return [parsed.strip()] if parsed.strip() else []
    except Exception:
        pass

    # Fallback parser
    s = s.strip("[]")
    parts = re.split(r",\s*", s)
    out = []
    for p in parts:
        p = p.strip().strip("'\"")
        if p and p.lower() not in {"nan", "none", "null"}:
            out.append(p)
    return out

In [ ]:
# Apply cleaning
df["job_title_clean"] = df["job_title"].apply(clean_text)
df["job_description_clean"] = df["job_description"].apply(clean_text)
df["category_clean"] = df["category"].apply(
    lambda x: clean_text(x).lower() if pd.notna(x) else np.nan
)

# Lowercased/normalised versions for matching
df["job_title_norm"] = df["job_title_clean"].str.lower()
df["job_description_norm"] = df["job_description_clean"].str.lower()

# Parse skills
df["job_skill_list"] = df["job_skill_set"].apply(parse_skill_set)

# Remove duplicates within each job's skill list
df["job_skill_list"] = df["job_skill_list"].apply(
    lambda xs: sorted(set(xs), key=lambda s: s.lower())
)

# Normalised skills
df["job_skill_normalized_list"] = df["job_skill_list"].apply(
    lambda xs: sorted(set(
        normalize_skill(x) for x in xs if pd.notna(normalize_skill(x))
    ))
)

df["job_skill_count"] = df["job_skill_list"].apply(len)

In [ ]:
# Deduplication
before = len(df)
df = df.drop_duplicates(keep="first").copy()
print(f"Dropped exact duplicate rows: {before - len(df)}")

dups = df[df.duplicated(subset=["job_id"], keep=False)].copy()
if len(dups) > 0:
    dups.to_csv(OUT_DIR / "duplicate_job_ids.csv", index=False)

before = len(df)
df = df.drop_duplicates(subset=["job_id"], keep="first").copy()
print(f"Dropped duplicate job_id rows: {before - len(df)}")

# Near-duplicate content check
before = len(df)
df = df.drop_duplicates(
    subset=["job_title_clean", "job_description_clean", "category_clean"],
    keep="first"
).copy()
print(f"Dropped near-duplicate content rows: {before - len(df)}")

In [ ]:
# Missing / usability flags
df["has_title"] = df["job_title_clean"].notna() & (df["job_title_clean"].str.len() > 0)
df["has_description"] = df["job_description_clean"].notna() & (df["job_description_clean"].str.len() > 0)
df["has_skills"] = df["job_skill_list"].apply(len) > 0

df["usable_for_semantic"] = df["has_description"]
df["usable_for_rule"] = df["has_skills"]
df["usable_for_kg"] = df["has_title"] & df["has_skills"]

unusable = df[~df["has_title"] & ~df["has_description"] & ~df["has_skills"]].copy()
unusable.to_csv(OUT_DIR / "unusable_jobs.csv", index=False)

print("Usable for semantic:", df["usable_for_semantic"].sum())
print("Usable for rule-based:", df["usable_for_rule"].sum())
print("Usable for KG:", df["usable_for_kg"].sum())

In [ ]:
# Long-format job-skill table
skills_long = df.explode("job_skill_list").dropna(subset=["job_skill_list"]).copy()
skills_long["skill_original"] = skills_long["job_skill_list"].astype(str).str.strip()
skills_long["skill_normalized"] = skills_long["skill_original"].apply(normalize_skill)

skills_long = skills_long[skills_long["skill_normalized"].notna()].copy()
skills_long = skills_long[
    ["job_id", "category_clean", "job_title_clean", "skill_original", "skill_normalized"]
].drop_duplicates(subset=["job_id", "skill_normalized"])

In [ ]:
# Skill vocabulary
skill_vocab = (
    skills_long.groupby("skill_normalized")
    .agg(
        frequency=("job_id", "nunique"),
        example_original=("skill_original", "first")
    )
    .reset_index()
    .sort_values(["frequency", "skill_normalized"], ascending=[False, True])
)

skill_vocab["esco_skill_id"] = None
skill_vocab["mapping_status"] = "unmapped"
skill_vocab["mapping_notes"] = None

# Mapping templates
esco_mapping = pd.DataFrame(columns=[
    "skill_normalized",
    "esco_skill_id",
    "esco_preferred_label",
    "mapping_status",
    "source",
    "notes"
])
esco_mapping.to_csv(OUT_DIR / "esco_skill_mapping_template.csv", index=False)

kenya_vocab = pd.DataFrame(columns=[
    "informal_term",
    "formal_term",
    "esco_occupation_uri",
    "esco_skill_uri",
    "source",
    "validated"
])
kenya_vocab.to_csv(OUT_DIR / "kenya_informal_vocab_template.csv", index=False)

In [ ]:
# Save processed job data
jobs_clean = df.copy()
jobs_clean["job_skill_list_json"] = jobs_clean["job_skill_list"].apply(json.dumps)
jobs_clean["job_skill_normalized_list_json"] = jobs_clean["job_skill_normalized_list"].apply(json.dumps)

jobs_clean.drop(
    columns=["job_skill_list", "job_skill_normalized_list"],
    errors="ignore"
).to_csv(OUT_DIR / "jobs_clean.csv", index=False)

skills_long.to_csv(OUT_DIR / "job_skills_long.csv", index=False)
skill_vocab.to_csv(OUT_DIR / "skill_vocab.csv", index=False)


In [ ]:
# Report output 
with open(OUT_DIR / "preprocessing_report.txt", "w", encoding="utf-8") as f:
    f.write("JOB CORPUS PREPROCESSING REPORT\n")
    f.write("=" * 40 + "\n")
    f.write(f"Original shape: {df.shape}\n")
    f.write(f"Final job rows: {len(jobs_clean)}\n")
    f.write(f"Long skill rows: {len(skills_long)}\n")
    f.write(f"Unique normalized skills: {skill_vocab.shape[0]}\n")
    f.write(f"Usable for semantic: {df['usable_for_semantic'].sum()}\n")
    f.write(f"Usable for rule-based: {df['usable_for_rule'].sum()}\n")
    f.write(f"Usable for KG: {df['usable_for_kg'].sum()}\n")
    f.write("\nMissing values after cleaning:\n")
    f.write(str(jobs_clean[["job_title_clean", "job_description_clean", "category_clean"]].isna().sum()))

print("\nDone. Files written to:", OUT_DIR.resolve())